# The follwoign codes focus on the extraction of the planned

1) Load waveform metadata (focus on PPG cases).

2) Extract height, weight, and heart rhythm information from CHARTEVENTS.

3) Save all extracted events into a single file.

4) Apply rules to clean events (minimum/maximum window length).

5) Add RESP channel information when available.

6) Extract WFDB signal segments (up to 15 min before each event).

7) Merge all per-record results into one dataframe.


## Prperation of the seletcion table

# Load wfs metadata

In [ ]:
wfs_headers_filename = "wfs_headers.pkl"

df_wfs=pd.read_pickle(wfs_headers_filename)

#select only samples with ppg
df_wfs = df_wfs[df_wfs.channels.apply(lambda x: "PLETH" in x)]

#convert to compatible subject id
df_wfs["subject_id"]=df_wfs["patient"].apply(lambda x: int(re.sub(r'^p0*', '', x)))
wfs_subject_ids = df_wfs.subject_id.unique()

#convert to datetime
df_wfs["start"]=pd.to_datetime(df_wfs["start"])
df_wfs["end"]=pd.to_datetime(df_wfs["end"])

#sort by subjects
df_wfs=df_wfs.sort_values(by="subject_id",ascending=True)

## Extraction of  Weight & height & heart rhythm using CHARTEVENTS.csv

In [ ]:
# --- Constants ---
# weight and height per subject and  ICU stay

# We note that several different codes exist for height and weight in the dataset. However, after preliminary analysis, we determined that only a subset of these codes is required for our final dataset. Additionally, we excluded ITEMID 226730 for height, as it represents duplicated information recorded in a different unit (please check https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/demographics/heightweight.sql) 

height_ids = [226707] 
weight_ids = [224639, 226512, 226846] 
unit_conversion_dict = {
    226707: 2.54,
}
heartrhythm_ids = [212,220048]

def find_closest_date(df, target_date, col_time="CHARTTIME"):
    # Convert target_date to datetime if it's not already
    target_date = pd.to_datetime(target_date)
    
    # Calculate the difference between each date and the target date
    df['date_diff'] = abs(df[col_time] - target_date)
    
    # Find the index of the minimum difference
    closest_index = df['date_diff'].idxmin()
    
    # Get the row with the closest date
    closest_row = df.loc[closest_index]
    
    # Drop the temporary 'date_diff' column
    df.drop('date_diff', axis=1, inplace=True)
    
    return closest_row

In [ ]:
res=[]
prev_subject_id=-1#avoid reloading the in-memory dataframe
missed_subject_ids = []

# height and weight bounds based on the references in the literature (e.g. https://onlinelibrary.wiley.com/doi/epdf/10.1002/oby.23687, https://link.springer.com/article/10.1186/1471-2288-13-38, https://link.springer.com/article/10.1186/s12884-015-0504-5)   
HEIGHT_MIN = 120 #cm
HEIGHT_MAX = 230 #cm
WEIGHT_MIN = 35 #kg
WEIGHT_MAX = 250 #kg

#OPTIONAL: SPLIT THIS LOOP UP INTO CHUNKS FOR PARALLEL PROCESSING
for idx,row in tqdm(df_wfs.iterrows(),total=len(df_wfs)):
    subject_id = row["subject_id"]
    
    
    if(subject_id!=prev_subject_id):
        
        # Load chart events for this subject.
        #
        # note
        # These files are not included in the repository  due to PhysioNet / MIMIC data usage restrictions.
        # Users must extract them from the MIMIC-III CHARTEVENTS table and split the data by SUBJECT_ID.
        #
        # Each file should be named:
        #     chunk_<subject_id>.csv
        #
        # and contain at least the following columns from CHARTEVENTS:
        #     SUBJECT_ID, HADM_ID, ICUSTAY_ID, CHARTTIME, ITEMID, VALUE, CGID
        #
        # This avoids repeatedly scanning the full CHARTEVENTS table (~330M rows)
        # and significantly speeds up the pipeline.



        filename_events = Path(mimic_filename_chunks_path)/("chunk_"+str(subject_id)+".csv")
        
        if(not filename_events.exists()):
            missed_subject_ids.append(subject_id)#print("Subject_ID",subject_id,"missing. Skipping.")
            continue
        # Filter the rows where the specified column has the desired value
        df_mimic_selection = pd.read_csv(filename_events)
       
        df_mimic_selection["CHARTTIME"]=pd.to_datetime(df_mimic_selection["CHARTTIME"])
        #height
        df_mimic_selection_height = df_mimic_selection[df_mimic_selection["ITEMID"].apply(lambda x: x in height_ids)].copy()
        df_mimic_selection_height = df_mimic_selection_height[df_mimic_selection_height["VALUE"].notna()].copy()
        df_mimic_selection_height["VALUE"] = df_mimic_selection_height.apply(
            lambda row: unit_conversion_dict[row["ITEMID"]]*row["VALUE"]
            if (row["ITEMID"] in unit_conversion_dict.keys() and isinstance(row["VALUE"],float))
            else row["VALUE"],
            axis=1
        )
        # ---- physiological filtering ----
        df_mimic_selection_height.loc[
            ~df_mimic_selection_height["VALUE"].between(HEIGHT_MIN, HEIGHT_MAX),
            "VALUE"
        ] = np.nan
        df_mimic_selection_height = df_mimic_selection_height[df_mimic_selection_height["VALUE"].notna()]        
        #weight
        df_mimic_selection_weight = df_mimic_selection[df_mimic_selection["ITEMID"].apply(lambda x: x in weight_ids)].copy()
        df_mimic_selection_weight = df_mimic_selection_weight[df_mimic_selection_weight["VALUE"].notna()].copy()
        # ---- physiological filtering ----
        df_mimic_selection_weight.loc[
            ~df_mimic_selection_weight["VALUE"].between(WEIGHT_MIN, WEIGHT_MAX),
            "VALUE"
        ] = np.nan

        df_mimic_selection_weight = df_mimic_selection_weight[df_mimic_selection_weight["VALUE"].notna()]
        df_mimic_selection = df_mimic_selection[df_mimic_selection["ITEMID"].apply(lambda x: x in heartrhythm_ids)].copy()#heart rhythm
        df_mimic_selection = df_mimic_selection[df_mimic_selection["VALUE"].notna()].copy() #remove nans
        df_mimic_selection=df_mimic_selection.sort_values(by="CHARTTIME",ascending=True)
        df_mimic_selection=df_mimic_selection.reset_index(drop=True)
        
        
        prev_subject_id = subject_id
        #print(subject_id, len(df_mimic_selection))
    
    if(len(df_mimic_selection)>0):#potential match
        
        df_mimic_selection["contains_event"]=df_mimic_selection.CHARTTIME.apply(lambda x: x>=row["start"] and x<=row["end"])
       
        for event_id,event_row in df_mimic_selection.iterrows():
            if(event_row["contains_event"]):
                has_previous = False if event_id==0 else df_mimic_selection.iloc[event_id-1]["HADM_ID"]==event_row["HADM_ID"]
                tmp={"record":row["record"],"subject_id":row["subject_id"],"channels":row["channels"],"freq":row["freq"],"start":row["start"],"end":row["end"]}
                
                tmp["hadm_id"]=event_row["HADM_ID"]
                tmp["icustay_id"]=event_row["ICUSTAY_ID"]
                                         
                tmp["event_time"]=event_row["CHARTTIME"]
                tmp["event_rhythm"]=event_row["VALUE"]
                tmp["event_cgid"]=event_row["CGID"]
                tmp["event_itemid"]=event_row["ITEMID"]
                if(len(df_mimic_selection_weight)>0):
                    tmp2= find_closest_date(df_mimic_selection_weight, event_row["CHARTTIME"])
                    tmp["weight"]=tmp2["VALUE"]
                else:
                    tmp["weight"]=np.nan
                if(len(df_mimic_selection_height)>0):
                    tmp2= find_closest_date(df_mimic_selection_height, event_row["CHARTTIME"])
                    tmp["height"]= tmp2["VALUE"]
                else:
                    tmp["height"]=np.nan
                
                if(has_previous):
                    tmp["prev_event_time"]=df_mimic_selection.iloc[event_id-1]["CHARTTIME"]
                    tmp["prev_event_rhythm"]=df_mimic_selection.iloc[event_id-1]["VALUE"]
                else:
                    tmp["prev_event_time"]=pd.NaT
                    tmp["prev_event_rhythm"]="N/A"
                res.append(tmp)
                                         
if(len(missed_subject_ids)):
    missed_subject_ids = np.unique(missed_subject_ids)
    print(len(missed_subject_ids), "subject ids missing.")


df.to_pickle("mimic_af_events.pkl")

## Adding age and sex from patients

In [ ]:
hadm_ids = df_events.hadm_id.unique()
subject_ids = df_events.subject_id.unique()

df_items = pd.read_csv("D_ITEMS.csv.gz")
#dischare diags
df_icd= pd.read_csv("DIAGNOSES_ICD.csv.gz")
df_icd_txt= pd.read_csv("D_ICD_DIAGNOSES.csv.gz")
df_icd2=df_icd.groupby("HADM_ID")["ICD9_CODE"].apply(lambda x: list(x)).to_frame()
from icdmappings import Mapper
mapper=Mapper()
df_icd2["ICD10_CODE"]=df_icd2["ICD9_CODE"].apply(lambda x: mapper.map(x, source='icd9', target='icd10'))
df_icd2["HADM_ID"]=df_icd2.index
df_icd2= df_icd2[df_icd2["HADM_ID"].apply(lambda x: x in hadm_ids)].copy()
def prepare_consistency_mapping(codes_unique, codes_unique_all, propagate_all=False):
    res={}
    for c in codes_unique:
        if(propagate_all):
            res[c]=[c[:i] for i in range(3,len(c)+1)]
        else:#only propagate if categories are already present
            res[c]=np.intersect1d([c[:i] for i in range(3,len(c)+1)],codes_unique_all)
    return res
codes_unique = [x for xs in list(df_icd2.ICD10_CODE) for x in xs]
codes_unique = [x for x in codes_unique if isinstance(x,str)]
icd_mapping = prepare_consistency_mapping(codes_unique, codes_unique, propagate_all=True)
df_icd2["ICD10_CODE"]=df_icd2["ICD10_CODE"].apply(lambda x: [y for y in x if isinstance(y,str)])
df_icd2["ICD10_CODE"]=df_icd2["ICD10_CODE"].apply(lambda x2: list(set([x for xs in [icd_mapping[y] for y in x2 ] for x in xs])))
#truncate to 3 digits and remove NoD
df_icd2["icd10_truncated"]=df_icd2["ICD10_CODE"].apply(lambda x: list(set([y[:3] for y in x if y!="NoD"])))

df_events=df_events.join(df_icd2["icd10_truncated"],on="hadm_id",how="left")

df_patients = pd.read_csv("PATIENTS.csv.gz")
df_patients = df_patients[df_patients["SUBJECT_ID"].apply(lambda x: x in subject_ids)]
df_patients["gender"]=df_patients["GENDER"]#rename/copy gender to lowercase
df_events=df_events.join(df_patients[["gender","DOB","SUBJECT_ID"]].set_index("SUBJECT_ID"),on="subject_id",how="left")

df_events["DOB"]=pd.to_datetime(df_events["DOB"])
df_events["age"]=df_events.apply(lambda row: np.floor((row["start"].to_pydatetime()-row["DOB"].to_pydatetime()).days/365.2425)
,axis=1)
#move ages in the 300s down to 91
df_events["age"]=df_events["age"].apply(lambda x: x if x<100 else 91)

df_events.drop("DOB",axis=1,inplace=True)
df_events.to_pickle("mimic_af_events_w_metadata.pkl")


## Preprationion steps for extracting the coressponding signals from Original WFDB files

In [ ]:
import pandas as pd
df = pd.read_pickle('mimic_af_events.pkl')
df

print("Patients (with rhythm events):",len(df.subject_id.unique()))
df["event_rhyth_eq_prev_event_rhythm"]=(df["event_rhythm"]==df["prev_event_rhythm"])# event and previous event coincide (most conservative selection)
df["time_diff_prev_event"]=(df["event_time"]-df["prev_event_time"])#time difference previous event to event
df["time_diff_start"]= (df["event_time"]-df["start"])#time difference start time series to event
df["signal_length"]=df.apply(lambda x: min(x["time_diff_prev_event"],x["time_diff_start"]),axis=1)  #either start or previous event whatever comes first

df["signal_length_s"]=df["signal_length"].apply(lambda x:x.total_seconds())
df

#conservative truncations >=30s and truncate to 15 min
df.loc[df["signal_length_s"]>900,"signal_length_s"]=900#truncate to at most 15 min (more conservative choices later)
df=df[df["signal_length_s"]>=30].copy().reset_index(drop=True)#at least 30s (standard criterion for AF)
df

df.to_pickle('mimic_af_events_w_metadata_with_signalID_with_RESP.pkl')

In [ ]:
import pandas as pd

# Load the dataframe
df_af = pd.read_pickle("mimic_af_events_w_metadata_with_signalID_with_RESP.pkl")
# Extract X (everything before "_")
df_af["record_base"] = df_af["record"].str.split("_").str[0]
# Count unique X values
n_unique_base = df_af["record_base"].nunique()
print("Number of unique X records:", n_unique_base)

df_headers = wfs_headers
# Make sure no accidental duplicates in headers
df_headers_unique = df_headers[['record', 'patient']].drop_duplicates()  # not nessesary since there is no douplication but just to be sure 
# Merge
df_af = df_af.merge(
    df_headers_unique,
    on='record',
    how='left'
)
# Check if anything failed to merge
missing = df_af['patient'].isna().sum()
print(f"Missing patient entries after merge: {missing}")
df_af['patient_num'] = df_af['patient'].str.replace('p', '').astype(int)
print("Consistency ratio:",
      (df_af['patient_num'] == df_af['subject_id']).mean())
icustays = pd.read_csv("ICUSTAYS.csv")[["ICUSTAY_ID", "DBSOURCE"]]

df_af = df_af.merge(
    icustays,
    left_on="icustay_id",
    right_on="ICUSTAY_ID",
    how="left"
)

df_af.drop(columns=["ICUSTAY_ID"], inplace=True)
df_sys = df_af[df_af["DBSOURCE"].isin(["carevue", "metavision"])].copy()
df_sys.drop(columns=["signalID"], inplace=True)



# ======================================================
# 1️⃣ Load ICU table
# ======================================================

df_icu = pd.read_csv("ICUSTAYS.csv")

# Keep only necessary columns
df_icu_small = df_icu[["ICUSTAY_ID", "SUBJECT_ID"]].copy()

# ======================================================
# 2️⃣ Ensure matching data types
# ======================================================

df_sys["icustay_id"] = df_sys["icustay_id"].astype(int)
df_sys["subject_id"] = df_sys["subject_id"].astype(int)

df_icu_small["ICUSTAY_ID"] = df_icu_small["ICUSTAY_ID"].astype(int)
df_icu_small["SUBJECT_ID"] = df_icu_small["SUBJECT_ID"].astype(int)

# ======================================================
# 3️⃣ Merge on ICU stay ID
# ======================================================

merged = df_sys.merge(
    df_icu_small,
    left_on="icustay_id",
    right_on="ICUSTAY_ID",
    how="left"
)

# ======================================================
# 4️⃣ Check for missing ICU stays
# ======================================================

missing_icustay = merged["SUBJECT_ID"].isna().sum()
print("ICU stays not found in ICUSTAYS:", missing_icustay)

# ======================================================
# 5️⃣ Check subject consistency
# ======================================================

mismatch = merged[merged["subject_id"] != merged["SUBJECT_ID"]]

print("Number of subject_id mismatches:", len(mismatch))

# ======================================================
# 6️⃣ Strict boolean summary
# ======================================================

all_match = (merged["subject_id"] == merged["SUBJECT_ID"]).all()

print("All subject IDs consistent:", all_match)

# ======================================================
# 7️⃣ Optional: show mismatches if any
# ======================================================

if len(mismatch) > 0:
    print("\nExample mismatches:")
    print(mismatch[["subject_id", "SUBJECT_ID", "icustay_id"]].head())

# 1️⃣ Sort to ensure correct ordering
df_sys = df_sys.sort_values(["record", "event_time"]).copy()

# 2️⃣ Create event_id per record
df_sys["event_id"] = df_sys.groupby("record").cumcount()




In [ ]:
check = (
    df_sys
    .groupby(["record", "event_id"])["event_time"]
    .nunique()
    .reset_index()
)

# Find problematic cases
problematic = check[check["event_time"] > 1]

print("Number of (record, event_id) with multiple event_time values:", len(problematic))

## Number of (record, event_id) with multiple event_time values: 0
## so we will save the dataframe.


df_sys.to_pickle("mimic_af_events_w_metadata_with_signalID_with_RESP_new.pkl", protocol=4)


## Extracting the WFDB files (maximum 15m in before th evenets) from the original WFDB files

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm  # Import tqdm for progress tracking
import wfdb  # Ensure you have wfdb installed
from datetime import datetime, date, timedelta
import gc


def print_wfdb_metadata(input_record):
    """
    Print comprehensive metadata for a WFDB record.
    
    Parameters:
    -----------
    input_record : str
        Base name of the input record (without .hea extension)
    """
    # Read the record
    record = wfdb.rdrecord(input_record)
    
    # Print basic record information
   # print("### Record Metadata ###")
    #print(f"Record Name: {input_record}")
    
    # Basic record attributes
   # print("\n--- Basic Attributes ---")
    attributes = [
        'fs', 'n_sig', 'sig_len', 'base_time', 'base_date', 'base_datetime'
    ]
    for attr in attributes:
        print(f"{attr}: {getattr(record, attr, 'N/A')}")
    
    # Signal-specific metadata
   # print("\n--- Signal Details ---")
    for i in range(len(record.sig_name)):
       # print(f"\nSignal {i + 1}:")
       # print(f"  Name: {record.sig_name[i]}")
       # print(f"  Units: {record.units[i]}")
       # print(f"  ADC Gain: {record.adc_gain[i]}")
       # print(f"  Baseline: {record.baseline[i]}")
        print(f"  Format: {record.fmt[i]}")
    
    # Read header file directly to get raw information
    print("\n--- Raw Header File Contents ---")
    try:
        with open(f"{input_record}.hea", 'r') as f:
            print(f.read())
    except Exception as e:
        print(f"Could not read header file: {e}")
    
    # Additional metadata from rdheader
    print("\n--- Additional Header Information ---")
    try:
        header_info = wfdb.rdheader(input_record)
        print(header_info)
    except Exception as e:
        print(f"Could not read header information: {e}")

def extract_wfdb_signals(
    record=None,
    input_record=None,
    channels_to_keep=None,
    start_time=None,
    end_time=None,
    write_dir='',
    output_record=''
):

    """
    Extract specified channels and time range from a WFDB record and save a new WFDB file.

    Parameters:
    -----------
    input_record : str
        Base name of the WFDB record.
    channels_to_keep : list, optional
        List of signal names to keep.
    start_time : float, optional
        Start time in seconds.
    end_time : float, optional
        End time in seconds.
    write_dir : str
        Directory where the output WFDB file will be saved.
    output_record : str
        Name of the output record.

    Returns:
    --------
    str
        Name of the generated WFDB record.
    """
    try:
        if record is None:
            if input_record is None:
                raise ValueError("Either 'record' or 'input_record' must be provided.")
            record = wfdb.rdrecord(input_record)


        if channels_to_keep is None:
            channels_to_keep = record.sig_name

        
        
        keep_indices = [record.sig_name.index(name) for name in channels_to_keep if name in record.sig_name]

        if not keep_indices:
            print(f"⚠️ No matching channels found in {input_record}, skipping.")
            return None  # Skip processing for this file


        fs = record.fs  
        start_sample = int(start_time * fs) if start_time is not None else 0
        end_sample = int(end_time * fs) if end_time is not None else record.p_signal.shape[0]

        if start_sample >= end_sample:
            raise ValueError("❌ Invalid time range: start_time and end_time result in no data.")

        extracted_signals = record.p_signal[start_sample:end_sample, keep_indices]
        if extracted_signals.size == 0:
            raise ValueError("❌ Extracted signal is empty. Check time range and channels.")

        new_record_name = output_record.replace('.', '_').split('/')[-1]

        kept_sig_name = [record.sig_name[i] for i in keep_indices]
        kept_units = [record.units[i] for i in keep_indices]
        kept_fmt = [record.fmt[i] for i in keep_indices]
        kept_adcs = [record.adc_gain[i] for i in keep_indices]
        kept_baselines = [record.baseline[i] for i in keep_indices]

   #     kwargs = {}
    #    if record.base_time or record.base_date or record.base_datetime:
   #         time_offset = start_sample / fs
     #       if record.base_datetime:
      #          kwargs['base_datetime'] = record.base_datetime + timedelta(seconds=time_offset)
     #       elif record.base_time:
     #           adjusted_base_time = (datetime.combine(date.today(), record.base_time) + timedelta(seconds=time_offset)).time()
     #           kwargs['base_time'] = adjusted_base_time
     #       if record.base_date:
     #           kwargs['base_date'] = record.base_date

        wfdb.wrsamp(
            record_name=new_record_name,
            fs=fs,
            units=kept_units,
            sig_name=kept_sig_name,
            p_signal=extracted_signals,
            fmt=kept_fmt,
            adc_gain=kept_adcs,
            baseline=kept_baselines,
            write_dir=write_dir,
          #  **kwargs
        )

        #print(f"✅ Successfully saved {write_dir}/{new_record_name}.hea and {write_dir}/{new_record_name}.dat")
        return new_record_name

    except Exception as e:
        print(f"❌ Error extracting {input_record}: {e}")


def is_ppg_flat(ppg_segment, fs):
    """
    Check if the PPG segment is flat.
    
    Criteria:
    1. At least 4 consecutive samples are equal to the segment's min or max value.
    2. Any single value appears in more than fs consecutive samples.
    """
    min_val = np.min(ppg_segment)
    max_val = np.max(ppg_segment)

    consecutive_min = (ppg_segment == min_val).astype(int)
    consecutive_max = (ppg_segment == max_val).astype(int)
    # Use kernel size 4, check for >= 4
    if (np.max(np.convolve(consecutive_min, np.ones(4, dtype=int), mode='valid')) >= 4 or
        np.max(np.convolve(consecutive_max, np.ones(4, dtype=int), mode='valid')) >= 4):
        return True

    # Check for any value repeating in more than fs consecutive samples
    count = 1
    for i in range(1, len(ppg_segment)):
        if ppg_segment[i] == ppg_segment[i - 1]:
            count += 1
            if count > fs:
                return True
        else:
            count = 1

    return False


def extract_full_with_metadata(df_ids, channels_to_keep, output_dir, progress_callback=None):



    os.makedirs(output_dir, exist_ok=True)
  
    exported_segments = []

    for idx in tqdm(range(len(df_ids)), desc="Processing events", disable=not os.isatty(1)):

        row = df_ids.iloc[idx]
        record_suffix = row['record']
        event_idx = row['event_id']
        patient_idx=row['patient']
        subject_idx = row['subject_id']
        event_rhythm_idx= row['event_rhythm']
        start_record=row['start']
        end_record=row['end']
        patient_idx_str = str(patient_idx)
        patient_idx_folder = patient_idx_str[:3]
        input_record = (f"/fs/dss/work/axba1153/Mohammad/mimic-af/AWS-mimic-iii-clinical-database-1.4/"
                        f"{patient_idx_folder}/{patient_idx}/{record_suffix}")
        base_output_name = f"{record_suffix}_{event_idx}"

        record = None

        try:

            record = wfdb.rdrecord(input_record)
            start_time = pd.to_datetime(row["start"])
            fs = row['freq']
            total_record_length_s = len(record.p_signal) / fs

            try:
                ppg_index = record.sig_name.index("PLETH")
            except ValueError:
                print(f"❌ PLETH channel not found in record {record_suffix}. Skipping event {event_idx}.")
                continue
            
            
            signal_length_s = row['signal_length_s']
            event_time = pd.to_datetime(row["event_time"])
            event_s = (event_time - start_time).total_seconds()
            #print("event_s",event_s)
            segments_duration_used = min(signal_length_s, 900)
            segments_start_s = max(event_s - segments_duration_used, 0)
            segments_end_s = min(event_s, total_record_length_s)
            available_duration = segments_end_s - segments_start_s
            n_segments = int(available_duration // 30)
            if n_segments <= 0:
                print(f"⚠️ Not enough duration to split into 30s segments for {record_suffix}, event {event_idx}")
                continue

            for i in range(n_segments):
                segment_end_s = event_s - (i * 30)
                segment_start_s = segment_end_s - 30
                if segment_start_s < segments_start_s:
                    continue
                start_sample = int(segment_start_s * fs)
                #print("start_sample",start_sample)
                end_sample = int(segment_end_s * fs)
                #print("end_sample",end_sample)
                ppg_segment = record.p_signal[start_sample:end_sample, ppg_index]
                # Split the 30s segment into 3 sub-segments of 10s each and check each for flatness
                is_any_subwindow_flat = False
                for j in range(3):
                    sub_start = j * fs * 10
                    sub_end = (j + 1) * fs * 10
                    sub_ppg_segment = ppg_segment[int(sub_start):int(sub_end)]
                    if is_ppg_flat(sub_ppg_segment, fs):
                        is_any_subwindow_flat = True
                        break

                if is_any_subwindow_flat:
                    continue

                output_record = os.path.join(output_dir, f"{base_output_name}_{i}")
                extract_wfdb_signals(record=record,
                     channels_to_keep=channels_to_keep,
                     start_time=segment_start_s,
                     end_time=segment_end_s,

                     write_dir=output_dir,
                     output_record=output_record)

                #print(f"✅ Exported window {i} for {base_output_name}")
                start_segment_dt = event_time - timedelta(seconds=(i+1) * 30)
                exported_segments.append({
                    "subject_id": subject_idx,
                    "record": record_suffix,
                    "event_id": event_idx,
                    "segment_id": i,
                    "signal_file_name": f"{base_output_name}_{i}",
                    "start_segment": start_segment_dt,
                    "event_time": event_time,
                    "event_rhythm": event_rhythm_idx,
                    "patient":patient_idx,
                    "start_record":start_record,
                    "end_record":end_record,
                })
        except Exception as e:
            print(f"❌ Failed to process {record_suffix} event {event_idx}: {e}")

        finally:
            if record is not None:
                del record
            gc.collect()



        if progress_callback:
            progress_callback()
    
    return pd.DataFrame(exported_segments)




# Example usage:



df = pd.read_pickle('mimic_af_events_w_metadata_with_signalID_with_RESP_new.pkl')

task_id = int(os.environ.get("SLURM_ARRAY_TASK_ID", "0"))

chunk_size = 10000
start_idxx = task_id * chunk_size
end_idxx = min(start_idxx + chunk_size, len(df))

if start_idxx >= len(df):
    print(f"Task {task_id}: start index exceeds dataframe length. Exiting.")
    exit(0)

df_ids = df.iloc[start_idxx:end_idxx]

print(f"Task {task_id} | Processing rows {start_idxx}:{end_idxx} | Chunk size: {len(df_ids)}")

channels_to_keep = ['PLETH', 'II', 'ABP', 'RESP']

base_output_root = "path/to/output" #in our case the name is  ..../WFDB_files_with_utils_5_15min_30s_segmented_no_Flat_PPG_new_4
output_dir = os.path.join(base_output_root, f"task_{task_id}")

segments_metadata_df = extract_full_with_metadata(
    df_ids,
    channels_to_keep,
    output_dir
)

if not segments_metadata_df.empty:
    pkl_filename = os.path.join(
        base_output_root,
        f"WFDB_files_segmented_task_{task_id}.pkl"
    )
    segments_metadata_df.to_pickle(pkl_filename)
    print(f"Saved {pkl_filename}")

del segments_metadata_df
gc.collect()

 